# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Introduction</h2>


This notebook demonstrates a JAX implementation for predicting pet pawpularity scores.

Your task is to predict engagement with a pet's profile, *Pawpularity*, based on the photograph for that profile. The dataset for this competition comprises images.

In this notebook, we use a pretrained backbone model [ResNet18](https://arxiv.org/pdf/1512.03385) to first produce meaningful image embeddings, then the pawpularity score is predicted by a simple MLP layer.

## Your Tasks

You need to implement the following key components marked with **TODO**:

1. **Model structure after backbone** - Design your custom MLP head
2. **Forward pass** - Implement the forward pass in the model
3. **Loss calculation** - Implement the RMSE loss function
4. **One step training** - Implement the training step function
5. **One step validation** - Implement the validation step function


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Install Required Libraries</h2>


In [ ]:
%pip install --upgrade "jax[cuda12]" -q
%pip install flax -q
%pip install optax -q
%pip install --upgrade wandb -q
%pip install colorama -q
%pip install albumentations -q
%pip install opencv-python -q
%pip install flaxmodels -q


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Import Required Libraries 📚</h2>


In [ ]:
import os
import gc
import cv2
import copy
import time
import random
from PIL import Image

# For data manipulation
import numpy as np
import pandas as pd

# JAX and Flax Imports
import jax
import jax.numpy as jnp
from jax import random as jax_random
from jax import jit, grad, value_and_grad
from typing import Any, Tuple
import flax
from flax import linen as nn
from flax.training import train_state, checkpoints
import optax

# For pretrained ResNet18
import flaxmodels as fm

# Data loading
from torch.utils.data import Dataset, DataLoader

# Utils
import joblib
from tqdm import tqdm
from collections import defaultdict
from functools import partial

# Sklearn Imports
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

# Albumentations for augmentations
import albumentations as A

# For colored terminal text
from colorama import Fore, Back, Style
c_ = Fore.CYAN
sr_ = Style.RESET_ALL

import warnings
warnings.filterwarnings("ignore")

# Check available JAX devices
print(f"JAX devices: {jax.devices()}")


# Load Datasets


In [ ]:
# If you are using colab, upload PA5 folder into your google drive, and load the notebook from it
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    petfinder_pawpularity_score_path = '/content/drive/MyDrive/PA4_train_and_test_data_fixed' # change this to your own path
# if you are running the notebook on server/local, specify the path directly
else:
    petfinder_pawpularity_score_path = '../PA5/PA4_train_and_test_data_fixed' # change this to your local path


<img src="https://i.imgur.com/gb6B4ig.png" width="400" alt="Weights & Biases" />

<span style="color: #000508; font-family: Segoe UI; font-size: 1.2em; font-weight: 300;"> Weights & Biases (W&B) is a set of machine learning tools that helps you build better models faster.</span>


In [ ]:
import wandb
import os

api_key = '54e65fed51afb0a477c937340023a6a5956284b4' # replace with your own wandb api key here
assert api_key != 'secret', 'If you want to use your W&B account, go to Add-ons -> Secrets and provide your W&B access token. \nGet your W&B access token from here: https://wandb.ai/authorize'
wandb.login(key=api_key)
anony = None
use_wandb = True


In [ ]:
ROOT_DIR = petfinder_pawpularity_score_path
TRAIN_DIR = petfinder_pawpularity_score_path + "/train"
TEST_DIR = petfinder_pawpularity_score_path + "/test"


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Training Configuration ⚙️</h2>

Adjust important hyper parameters here


In [ ]:
CONFIG = dict(
    seed = 42,
    backbone = 'resnet18',
    freeze_backbone = True, # freeze pre-trained weights for fine-tuning
    train_batch_size = 16,
    valid_batch_size = 32,
    img_size = 224, # standard size for resnet18
    epochs = 3,
    learning_rate = 1e-4,
    scheduler = 'cosine',  # 'cosine' or 'constant'
    min_lr = 1e-6,
    T_max = 100,
    weight_decay = 1e-6,
    n_accumulate = 1,
    num_classes = 1,
    competition = 'PetFinder',
    _wandb_kernel = 'deb'
)


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Set Seed for Reproducibility</h2>


In [ ]:
def set_seed(seed = 42):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    random.seed(seed)
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Note: JAX uses explicit PRNG keys, initialized in training code

set_seed(CONFIG['seed'])


In [ ]:
def get_train_file_path(id):
    return f"{TRAIN_DIR}/{id}.jpg"


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Read the Data 📖</h2>


In [ ]:
df = pd.read_csv(f"{ROOT_DIR}/train.csv")
df['file_path'] = df['Id'].apply(get_train_file_path)


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Dataset Class</h2>


In [ ]:
class PawpularityDataset(Dataset):
    def __init__(self, root_dir, df, transforms=None):
        self.root_dir = root_dir
        self.df = df
        self.file_names = df['file_path'].values
        self.targets = df['Pawpularity'].values
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_path = self.file_names[index]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        target = self.targets[index]

        if self.transforms:
            img = self.transforms(image=img)["image"]

        # Ensure img is numpy array (albumentations might return numpy already)
        if not isinstance(img, np.ndarray):
            img = np.array(img)

        # Convert to float32 and ensure channels-last format (H, W, C)
        if img.dtype != np.float32:
            img = img.astype(np.float32)

        return img, np.float32(target)


# Data Augmentations


In [ ]:
# Note: We don't use ToTensorV2 for JAX - we'll handle conversion in the dataset
data_transforms = {
    "train": A.Compose([
        A.Resize(CONFIG['img_size'], CONFIG['img_size']),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ], p=1.),

    "valid": A.Compose([
        A.Resize(CONFIG['img_size'], CONFIG['img_size']),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ], p=1.)
}


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Create Model</h2>


In [ ]:
class PawpularityModel(nn.Module):
    """
    Model using pretrained ResNet18 backbone with custom MLP head.

    TODO: Implement the model structure after the backbone.
    You need to:
    1. Extract features from ResNet18 backbone
    2. Design your custom MLP head with:
       - At least one hidden layer
       - ReLU activation
       - Output layer for regression (num_classes=1)

    Hint: Use flaxmodels ResNet18 with output='activations' and pretrained='imagenet'

    Inputs to __call__:
        x: jnp.ndarray, shape (B, H, W, C)
           - Batch of images in channels-last format
           - B: batch size
           - H: image height (e.g., 224)
           - W: image width (e.g., 224)
           - C: number of channels (3 for RGB)
           - Data type: float32, values normalized to [0, 1] or ImageNet stats

        training: bool
           - Whether the model is in training mode (affects dropout, batch norm)
           - True during training, False during validation/inference

    Returns:
        output: jnp.ndarray, shape (B, num_classes)
           - Model predictions for pawpularity scores
           - B: batch size
           - num_classes: 1 for regression
           - Data type: float32
    """
    num_classes: int = 1

    @nn.compact
    def __call__(self, x, training: bool = True):
        # x shape: (B, H, W, C) - channels-last format for JAX

        # Use pretrained ResNet18 backbone
        # flaxmodels ResNet18 expects channels-last format
        # When output='activations', it returns a dictionary of activations.
        resnet_outputs = fm.ResNet18(output='activations', pretrained='imagenet', normalize=False)
        all_features_dict = resnet_outputs(x, train=training)

        # Extract the features from the last convolutional block for global average pooling.
        # For ResNet18, 'block4_1' typically holds the last spatial feature map.
        features = all_features_dict['block4_1']

        # TODO: Design your custom MLP head
        # Hint: Use nn.Dense layers with nn.relu activation
        # Example structure: hidden -> relu -> output
        # Note: features may be shape (B, C) or (B, H, W, C) depending on ResNet18 output
        x = jnp.mean(features, axis=(1, 2))
        x = nn.Dense(features=256)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        output = nn.Dense(features=self.num_classes)(x)


        # TODO: Return the final output
        return output

# Initialize the model
model = PawpularityModel(num_classes=CONFIG['num_classes'])


In [ ]:
# Test dummy forward pass
# Initialize model with random key and dummy input
rng = jax_random.PRNGKey(CONFIG['seed'])
rng, init_rng = jax_random.split(rng)

# JAX uses channels-last format: (B, H, W, C)
dummy_input = jnp.ones((1, CONFIG['img_size'], CONFIG['img_size'], 3))

# Initialize model parameters
variables = model.init(init_rng, dummy_input, training=False)
params = variables['params']
batch_stats = variables.get('batch_stats', {})

print(f"Model initialized successfully!")
print(f"Number of parameters: {sum(x.size for x in jax.tree_util.tree_leaves(params))}")

# Test forward pass
output = model.apply({'params': params, 'batch_stats': batch_stats},
                     dummy_input, training=False)
print(f"\nOutput shape: {output.shape}")
print(f"Output: {output}")


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Loss Function</h2>


In [ ]:
def rmse_loss(predictions, targets):
    """
    TODO: Implement Root Mean Squared Error loss

    Steps:
    1. Reshape predictions and targets to 1D arrays (flatten them)
    2. Compute the mean squared error: mse = mean((predictions - targets)^2)
    3. Return the square root of MSE: sqrt(mse)

    Hint: Use jnp.mean and jnp.sqrt for JAX operations

    Inputs:
        predictions: jnp.ndarray, shape (B, 1) or (B,)
           - Model predictions for pawpularity scores
           - B: batch size
           - Data type: float32
           - Values can be any real number (not necessarily in [0, 100])

        targets: jnp.ndarray, shape (B, 1) or (B,)
           - Ground truth pawpularity scores
           - B: batch size (must match predictions)
           - Data type: float32
           - Values typically in range [0, 100]

    Returns:
        loss: jnp.ndarray, scalar (shape ())
           - Root Mean Squared Error loss value
           - Data type: float32
           - Always non-negative
    """
    # TODO: Implement RMSE loss below
    predictions = predictions.ravel()
    targets = targets.ravel()
    mse = jnp.mean(jnp.square(predictions - targets))
    return jnp.sqrt(mse)


# <h2 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">Training Function</h2>


In [ ]:
# Helper function to convert numpy batch to JAX format
def numpy_collate(batch):
    """Convert PyTorch DataLoader batch to JAX format"""
    images, targets = zip(*batch)
    # Stack and convert to JAX arrays
    images = jnp.array(np.stack(images))
    targets = jnp.array(np.array(targets))

    # Ensure channels-last format (B, H, W, C)
    if len(images.shape) == 4 and images.shape[1] == 3:
        # If channels-first, convert to channels-last
        images = jnp.transpose(images, (0, 2, 3, 1))

    return images, targets


@jit
def train_step(state, batch, dropout_rng):
    """
    TODO: Implement one step of training

    This function should:
    1. Define a loss function that:
       - Performs forward pass with training=True
       - Computes loss using rmse_loss
       - Returns loss and (new_variables, predictions) as auxiliary output
    2. Compute gradients using value_and_grad
    3. Update model parameters using state.apply_gradients
    4. Update batch_stats from new_variables
    5. Return updated state, loss, and predictions

    Hint: Use state.apply_fn with mutable=['batch_stats'] for training

    Inputs:
        state: TrainState object
           - Contains model parameters, optimizer state, and batch statistics
           - state.params: dict, model parameters (weights and biases)
           - state.batch_stats: dict, batch normalization statistics
           - state.apply_fn: function, model forward pass function
           - state.opt_state: optimizer state (for AdamW)

        batch: tuple (images, targets)
           - images: jnp.ndarray, shape (B, H, W, C)
             * Batch of images in channels-last format
             * B: batch size, H: height (224), W: width (224), C: channels (3)
             * Data type: float32, normalized
           - targets: jnp.ndarray, shape (B,) or (B, 1)
             * Ground truth pawpularity scores
             * B: batch size (must match images)
             * Data type: float32, values in [0, 100]

        dropout_rng: jax.random.PRNGKey
           - Random number generator key for dropout layers
           - Used to ensure reproducible randomness during training
           - Type: jax.random.PRNGKey

    Returns:
        state: TrainState object
           - Updated training state with new parameters and batch stats
           - Parameters have been updated via gradient descent
           - Batch stats have been updated for BatchNorm layers

        loss: jnp.ndarray, scalar (shape ())
           - RMSE loss value for this batch
           - Data type: float32

        predictions: jnp.ndarray, shape (B, 1) or (B,)
           - Model predictions for this batch
           - Data type: float32
    """
    images, targets = batch

    def loss_fn(params):
        """
        TODO: Implement forward pass and loss calculation

        This inner function computes the loss for a given set of parameters.
        It's used by value_and_grad to compute gradients with respect to params.

        Inputs:
            params: dict (PyTree structure)
               - Model parameters (weights and biases) for which to compute loss
               - Same structure as state.params
               - Contains all trainable parameters of the model
               - Data type: jnp.ndarray for each parameter tensor
               - Note: This function closes over 'images', 'targets', 'state.batch_stats',
                 'state.apply_fn', and 'dropout_rng' from the outer scope

        Returns:
            loss: jnp.ndarray, scalar (shape ())
               - RMSE loss value for this batch
               - Data type: float32
               - Always non-negative

            (new_variables, predictions): tuple
               - new_variables: dict
                 * Contains updated batch statistics from BatchNorm layers
                 * Structure: {'batch_stats': {...}}
                 * Only includes mutable variables (batch_stats)
                 * Used to update running statistics during training
               - predictions: jnp.ndarray, shape (B, 1) or (B,)
                 * Model predictions for this batch
                 * B: batch size (must match images)
                 * Data type: float32
                 * Used for logging and debugging

        Note: This function must return the loss as the first element and
        (new_variables, predictions) as auxiliary output for value_and_grad to work correctly.
        """
        # TODO: Implement forward pass and loss calculation
        predictions, new_variables = state.apply_fn(
            {'params': params, 'batch_stats': state.batch_stats},
            images,
            training=True,
            mutable=['batch_stats'],
            rngs={'dropout': dropout_rng}
        )
        loss = rmse_loss(predictions, targets)
        return loss, (new_variables, predictions)



    # TODO: Compute gradients using value_and_grad
    grad_fn = jax.value_and_grad(loss_fn, argnums=0, has_aux=True)
    (loss, (new_variables, predictions)), grads = grad_fn(state.params)

    # TODO: Update parameters
    state = state.apply_gradients(grads=grads)

    # TODO: Update batch stats
    state = state.replace(batch_stats=new_variables['batch_stats'])

    # TODO: Return state, loss, predictions
    return state, loss, predictions




def train_one_epoch(state, dataloader, epoch, rng):
    """Train for one epoch"""
    running_loss = 0.0
    dataset_size = 0

    bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch_data in bar:
        # batch_data is already processed by numpy_collate via DataLoader's collate_fn
        images, targets = batch_data # <--- Removed redundant call to numpy_collate

        # Split RNG for dropout
        rng, dropout_rng = jax_random.split(rng)

        # Training step
        state, loss, predictions = train_step(state, (images, targets), dropout_rng)

        # Bookkeeping
        batch_size = images.shape[0]
        running_loss += float(loss) * batch_size
        dataset_size += batch_size
        epoch_loss = running_loss / dataset_size

        # Get current learning rate
        if hasattr(state.opt_state, 'hyperparams'):
            current_lr = float(state.opt_state.hyperparams['learning_rate'])
        else:
            current_lr = CONFIG['learning_rate']

        bar.set_postfix(Epoch=epoch, Train_Loss=f"{epoch_loss:.4f}", LR=f"{current_lr:.6f}")

        if step % 10 == 0:  # Log every 10 steps to avoid too much overhead
            wandb.log({"Train Loss": epoch_loss, "LR": current_lr})

    return state, epoch_loss, rng

In [ ]:
@jit
def eval_step(state, batch):
    """
    TODO: Implement one step of validation

    This function should:
    1. Perform forward pass with training=False (no dropout, use running batch stats)
    2. Compute loss using rmse_loss
    3. Return loss and predictions

    Hint: Use state.apply_fn with training=False (no mutable needed for eval)

    Inputs:
        state: TrainState object
           - Contains model parameters, optimizer state, and batch statistics
           - state.params: dict, model parameters (weights and biases)
           - state.batch_stats: dict, batch normalization statistics (used but not updated)
           - state.apply_fn: function, model forward pass function

        batch: tuple (images, targets)
           - images: jnp.ndarray, shape (B, H, W, C)
             * Batch of images in channels-last format
             * B: batch size, H: height (224), W: width (224), C: channels (3)
             * Data type: float32, normalized
           - targets: jnp.ndarray, shape (B,) or (B, 1)
             * Ground truth pawpularity scores
             * B: batch size (must match images)
             * Data type: float32, values in [0, 100]

    Returns:
        loss: jnp.ndarray, scalar (shape ())
           - RMSE loss value for this batch
           - Data type: float32
           - Always non-negative

        predictions: jnp.ndarray, shape (B, 1) or (B,)
           - Model predictions for this batch
           - Data type: float32
           - Used for computing validation metrics
    """
    images, targets = batch

    # TODO: Implement forward pass in eval mode
    predictions = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        images,
        training=False
    )

    # TODO: Compute loss
    loss = rmse_loss(predictions, targets)

    # TODO: Return loss and predictions
    return loss, predictions


def valid_one_epoch(state, dataloader, epoch):
    """Validation for one epoch"""
    dataset_size = 0
    running_loss = 0.0

    TARGETS = []
    PREDS = []

    bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch_data in bar:
        # batch_data is already processed by numpy_collate via DataLoader's collate_fn
        images, targets = batch_data # <--- Removed redundant call to numpy_collate

        batch_size = images.shape[0]

        # Evaluation step
        loss, predictions = eval_step(state, (images, targets))

        running_loss += float(loss) * batch_size
        dataset_size += batch_size

        epoch_loss = running_loss / dataset_size

        # Store predictions and targets
        PREDS.append(np.array(predictions.flatten()))
        TARGETS.append(np.array(targets.flatten()))

        bar.set_postfix(Epoch=epoch, Valid_Loss=f"{epoch_loss:.4f}")

        if step % 10 == 0:
            wandb.log({"Valid Loss": epoch_loss})

    TARGETS = np.concatenate(TARGETS)
    PREDS = np.concatenate(PREDS)
    val_rmse = root_mean_squared_error(TARGETS, PREDS)
    gc.collect()

    return epoch_loss, val_rmse

In [ ]:
# Custom TrainState with batch_stats for BatchNorm
class TrainState(train_state.TrainState):
    batch_stats: Any = None


def create_train_state(rng, model, learning_rate, weight_decay, total_steps=None):
    """Create initial training state"""
    # Create dummy input to initialize model
    dummy_input = jnp.ones((1, CONFIG['img_size'], CONFIG['img_size'], 3))

    # Initialize model
    variables = model.init(rng, dummy_input, training=False)
    params = variables['params']
    batch_stats = variables.get('batch_stats', {})

    # Create learning rate schedule
    if CONFIG['scheduler'] == 'cosine' and total_steps is not None:
        schedule = optax.cosine_decay_schedule(
            init_value=learning_rate,
            decay_steps=total_steps,
            alpha=CONFIG['min_lr'] / learning_rate
        )
    else:
        schedule = learning_rate

    # Create optimizer with weight decay
    tx = optax.adamw(learning_rate=schedule, weight_decay=weight_decay)

    # Create training state
    state = TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats
    )

    return state


def run_training(state, train_loader, valid_loader, num_epochs, rng):
    """Main training loop"""
    print(f"[INFO] Using JAX devices: {jax.devices()}\n")

    start = time.time()
    best_params = state.params
    best_batch_stats = state.batch_stats
    best_epoch_rmse = np.inf
    history = defaultdict(list)

    for epoch in range(1, num_epochs + 1):
        gc.collect()

        # Train for one epoch
        state, train_epoch_loss, rng = train_one_epoch(
            state, train_loader, epoch=epoch, rng=rng
        )

        # Validate
        val_epoch_loss, val_epoch_rmse = valid_one_epoch(
            state, valid_loader, epoch=epoch
        )

        history['Train Loss'].append(train_epoch_loss)
        history['Valid Loss'].append(val_epoch_loss)
        history['Valid RMSE'].append(val_epoch_rmse)

        # Log the metrics
        wandb.log({
            "Epoch Train Loss": train_epoch_loss,
            "Epoch Valid Loss": val_epoch_loss,
            "Epoch Valid RMSE": val_epoch_rmse
        })

        print(f'Epoch {epoch} Valid RMSE: {val_epoch_rmse:.4f}')

        # Save best model
        if val_epoch_rmse <= best_epoch_rmse:
            print(f"{c_}Validation RMSE Improved ({best_epoch_rmse:.4f} ---> {val_epoch_rmse:.4f})")
            best_epoch_rmse = val_epoch_rmse
            run.summary["Best RMSE"] = best_epoch_rmse

            # Save model checkpoint
            best_params = state.params
            best_batch_stats = state.batch_stats

            checkpoint_dir = os.path.abspath(f"./checkpoints")
            os.makedirs(checkpoint_dir, exist_ok=True)
            checkpoint_path = f"{checkpoint_dir}/RMSE{best_epoch_rmse:.4f}_epoch{epoch}"

            checkpoints.save_checkpoint(
                ckpt_dir=checkpoint_path,
                target={'params': state.params, 'batch_stats': state.batch_stats},
                step=epoch,
                overwrite=True
            )
            print(f"Model Saved to {checkpoint_path}{sr_}")

        print()

    end = time.time()
    time_elapsed = end - start
    print('Training complete in {:.0f}h {:.0f}m {:.0f}s'.format(
        time_elapsed // 3600, (time_elapsed % 3600) // 60, (time_elapsed % 3600) % 60))
    print("Best RMSE: {:.4f}".format(best_epoch_rmse))

    # Restore best model
    state = state.replace(params=best_params, batch_stats=best_batch_stats)

    return state, history


In [ ]:
from sklearn.model_selection import train_test_split
def prepare_loaders(test_size=0.2, random_state=42):
    # split dataframe into train/valid
    df_train, df_valid = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
        stratify=None  # Can't stratify on continuous target
    )

    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)

    # build datasets
    train_dataset = PawpularityDataset(TRAIN_DIR, df_train, transforms=data_transforms['train'])
    valid_dataset = PawpularityDataset(TRAIN_DIR, df_valid, transforms=data_transforms['valid'])

    # dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['train_batch_size'],
        shuffle=True,
        pin_memory=True,
        drop_last=True,
        collate_fn=numpy_collate # <--- Pass numpy_collate as the collate_fn
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CONFIG['valid_batch_size'],
        shuffle=False,
        pin_memory=True,
        collate_fn=numpy_collate # <--- Pass numpy_collate as the collate_fn
    )

    return train_loader, valid_loader

<span style="color: #000508; font-family: Segoe UI; font-size: 1.5em; font-weight: 300;">Create Dataloaders</span>


In [ ]:
train_loader, valid_loader = prepare_loaders()


<span style="color: #000508; font-family: Segoe UI; font-size: 1.5em; font-weight: 300;">Initialize Model and Training State</span>


In [ ]:
# Initialize random key
rng = jax_random.PRNGKey(CONFIG['seed'])
rng, init_rng = jax_random.split(rng)

# Calculate total steps for learning rate schedule
total_steps = len(train_loader) * CONFIG['epochs']

# Create the model
model_instance = PawpularityModel(num_classes=CONFIG['num_classes'])

# Create training state
state = create_train_state(
    init_rng,
    model_instance,
    CONFIG['learning_rate'],
    CONFIG['weight_decay'],
    total_steps=total_steps
)

print("Training state created successfully!")
print(f"Number of parameters: {sum(x.size for x in jax.tree_util.tree_leaves(state.params))}")


In [ ]:
run = wandb.init(project='Pawpularity',
                 config=CONFIG,
                 job_type='Train',
                 anonymous='must')


<span style="color: #000508; font-family: Segoe UI; font-size: 1.5em; font-weight: 300;">Start Training</span>


# **IMPORTANT**
If you are using google colab, pay attention to your runtime usage. Constantly save checkpoints and download the model weights. If your runtime dies, all information will be lost!


In [ ]:
state, history = run_training(
    state,
    train_loader,
    valid_loader,
    num_epochs=CONFIG['epochs'],
    rng=rng
)


In [ ]:
run.finish()


# Inference


In [ ]:
def get_test_file_path(id):
    return f"{TEST_DIR}/{id}.jpg"
df_test = pd.read_csv(f"{ROOT_DIR}/test.csv")
df_test['file_path'] = df_test['Id'].apply(get_test_file_path)
df_test.head()
class PawpularityTestDataset(Dataset):
    def __init__(self, root_dir, df, transforms=None):
        self.root_dir = root_dir
        self.df = df
        self.file_names = df['file_path'].values
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        img_path = self.file_names[index]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transforms:
            img = self.transforms(image=img)["image"]

        # Ensure img is numpy array
        if not isinstance(img, np.ndarray):
            img = np.array(img)

        # Convert to float32 and ensure channels-last format (H, W, C)
        if img.dtype != np.float32:
            img = img.astype(np.float32)

        return img



In [ ]:
import os

# Re-create the checkpoint directory path
checkpoint_dir = os.path.abspath("./checkpoints")

# Find the checkpoint folder with the lowest RMSE
checkpoint_folders = []
if os.path.exists(checkpoint_dir):
    checkpoint_folders = [
        d for d in os.listdir(checkpoint_dir)
        if os.path.isdir(os.path.join(checkpoint_dir, d)) and d.startswith("RMSE")
    ]

if checkpoint_folders:
    # Sort by RMSE (ascending) to get the best performing checkpoint first
    # Assuming format 'RMSE<float>_epoch<int>'
    checkpoint_folders.sort(key=lambda x: float(x.split('RMSE')[1].split('_')[0]))

    best_checkpoint_folder = checkpoint_folders[0]
    best_checkpoint_path = os.path.join(checkpoint_dir, best_checkpoint_folder)

    # Restore the state. The target structure must match what was saved.
    restored_data = checkpoints.restore_checkpoint(
        ckpt_dir=best_checkpoint_path,
        target={'params': state.params, 'batch_stats': state.batch_stats}
    )

    # Update the current state with the restored parameters and batch stats
    state = state.replace(params=restored_data['params'], batch_stats=restored_data['batch_stats'])

    print(f"Successfully loaded checkpoint from: {best_checkpoint_path}")
else:
    print(f"No checkpoints found in {checkpoint_dir}. Model will use current state.")

In [ ]:
test_dataset = PawpularityTestDataset(TEST_DIR, df_test, transforms=data_transforms['valid']) # using valid transforms for test

# Helper function to collate test images
def test_collate(batch):
    """Convert PyTorch DataLoader batch to JAX format for test data"""
    # Stack and convert to JAX arrays
    images = jnp.array(np.stack(batch))

    # Ensure channels-last format (B, H, W, C)
    if len(images.shape) == 4 and images.shape[1] == 3:
        # If channels-first, convert to channels-last
        images = jnp.transpose(images, (0, 2, 3, 1))

    return images

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['valid_batch_size'], # using valid batch size for consistency
    shuffle=False,
    pin_memory=True,
    collate_fn=test_collate
)

@jit
def inference_step(state, images):
    """
    Single inference step - JIT compiled for speed

    Inputs:
        state: TrainState object
           - Contains model parameters and batch statistics
           - state.params: dict, model parameters
           - state.batch_stats: dict, batch normalization statistics
           - state.apply_fn: function, model forward pass function

        images: jnp.ndarray, shape (B, H, W, C)
           - Batch of images in channels-last format
           - B: batch size, H: height (224), W: width (224), C: channels (3)
           - Data type: float32, normalized

    Returns:
        predictions: jnp.ndarray, shape (B, 1) or (B,)
           - Model predictions for pawpularity scores
           - Data type: float32
    """
    # Forward pass in eval mode
    variables = {'params': state.params, 'batch_stats': state.batch_stats}
    predictions = state.apply_fn(variables, images, training=False)
    return predictions


def inference_fn(state, dataloader):
    """
    Run inference on test data

    Inputs:
        state: TrainState object
           - Contains model parameters and batch statistics from training
           - Should contain the best model weights after training

        dataloader: DataLoader
           - Test data loader with images
           - Images are already in JAX format (channels-last)

    Returns:
        predictions: np.ndarray, shape (N,)
           - Predictions for all test samples
           - N: total number of test samples
           - Data type: float32
           - Values are raw model outputs (may need clipping to [0, 100])
    """
    PREDS = []
    bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, images in bar:
        # images are already in JAX format from collate_fn
        # Forward pass
        predictions = inference_step(state, images)

        # Store predictions
        PREDS.append(np.array(predictions.flatten()))

    PREDS = np.concatenate(PREDS)
    return PREDS

predictions = inference_fn(state, test_loader)



In [ ]:
# Create submission file
submission_df = pd.DataFrame({'Id': df_test['Id'], 'Pawpularity': predictions})
# Pawpularity scores are between 0 and 100, so clip predictions
submission_df['Pawpularity'] = submission_df['Pawpularity'].clip(0, 100)
submission_df.head()
submission_df.to_csv('submission.csv', index=False) # uncomment to save the submission file


In [ ]:
sorted_df = submission_df.sort_values(by='Pawpularity', ascending=False)
sorted_df.head()

top_ids = sorted_df.head(3)['Id'].tolist()


In [ ]:
import matplotlib.pyplot as plt

# Get the file paths for the top 3 most popular pets
top_pet_paths = [get_test_file_path(id) for id in top_ids]

# Display the images
plt.figure(figsize=(15, 5))
for i, img_path in enumerate(top_pet_paths):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Pet ID: {top_ids[i]}\nPredicted Pawpularity: {sorted_df.iloc[i]['Pawpularity']:.2f}")
    plt.axis('off')
plt.tight_layout()
plt.show()
